# Pipeline Analysis

**Stage 5 of the benchmarking pipeline**: aggregates all method outputs
and produces Table 1 (per-method metrics) and Table 2 (complementarity).

Follows Appendix A pseudocode exactly.  Run *after* all method notebooks have finished.

```
Stage 1  Derive specs      <- prexsyn_baseline.ipynb  -> seeds_for_methods.json
Stage 2  PrexSyn seeds     <- prexsyn_baseline.ipynb
Stage 3  Method variants   <- per-method notebooks    -> *_scores.csv
Stage 4  Evaluation gates  <- per-method notebooks    -> *_synth_checkpoint.json
Stage 5  Aggregation       <- THIS NOTEBOOK           -> pipeline_results/
```

## Configuration

In [ ]:
import json
import pathlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

ROOT     = pathlib.Path('..')
DATA_DIR = ROOT / 'data'
OUT_DIR  = DATA_DIR / 'pipeline_results'   # <- all outputs written here
OUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT))
from src.evaluation.scoring_v2 import (
    classify_hits, summarize, complementarity, best_combinations,
    DEFAULT_TAU_T_LIST, DEFAULT_TAU_D,
)

TAU_T_LIST = DEFAULT_TAU_T_LIST   # [0.6, 0.7, 0.85]  <- paper §2.3
TAU_D      = DEFAULT_TAU_D        # 0.8

# Method registry --------------------------------------------------
# Outputs from data_generation.ipynb (data/generation/).
# Legacy paths (data/) kept as fallback comments.
GEN_DIR = DATA_DIR / 'generation'

METHODS = {
    'baseline': {
        'scores':     GEN_DIR / 'baseline/baseline_scores.csv',
        'synth_ckpt': GEN_DIR / 'baseline/synth_checkpoint.json',
    },
    'CReM': {
        'scores':     GEN_DIR / 'crem/crem_scores.csv',
        'synth_ckpt': GEN_DIR / 'crem/synth_checkpoint.json',
    },
    'LibINVENT': {
        'scores':     GEN_DIR / 'libinvent/libinvent_scores.csv',
        'synth_ckpt': GEN_DIR / 'libinvent/synth_checkpoint.json',
    },
    # 'mmpdb':   {'scores': GEN_DIR / 'mmpdb/mmpdb_scores.csv',   'synth_ckpt': GEN_DIR / 'mmpdb/synth_checkpoint.json'},
    # 'JT-VAE':  {'scores': GEN_DIR / 'jtvae/jtvae_scores.csv',   'synth_ckpt': GEN_DIR / 'jtvae/synth_checkpoint.json'},
    # 'ReactEA': {'scores': GEN_DIR / 'reactea/reactea_scores.csv','synth_ckpt': GEN_DIR / 'reactea/synth_checkpoint.json'},
}

SUMMARY_CSV    = OUT_DIR / 'summary.csv'            # Table 1
COMPLEMENT_CSV = OUT_DIR / 'complementarity.csv'    # Table 2 pairwise
COMBOS_CSV     = OUT_DIR / 'best_combinations.csv'  # Table 2 top combos

print(f'Output dir    : {OUT_DIR}')
print(f'Methods       : {list(METHODS)}')
print(f'Thresholds    : tau_t={TAU_T_LIST}, tau_d={TAU_D}')


---
## Stage 1 — Property Specifications (from ChEMBL)

**Pseudocode**: `spec_i = ExtractPropertySpec(m_i)` — ECFP4, MW, CLogP, TPSA,
HBD, HBA, RotBonds, 3D conformer via ETKDG.

Specifications were derived by `prexsyn_baseline.ipynb` and written to
`seeds_for_methods.json`.  We load them here for quality-bin stratification.

In [ ]:
assert (GEN_DIR / 'seeds_for_methods.json').exists(), (
    'seeds_for_methods.json missing. Run data_generation.ipynb first.'
)

with open(GEN_DIR / 'seeds_for_methods.json') as f:
    seeds = json.load(f)

seeds_df = pd.DataFrame([
    {
        'spec_smiles':      e['spec_smiles'],
        'seed_smiles':      e['seed_smiles'],
        'baseline_quality': e['baseline_quality'],
        'quality_bin':      e['quality_bin'],
    }
    for e in seeds
])

print(f'Specs loaded: {len(seeds_df)}')
print('\nQuality-bin counts:')
print(seeds_df['quality_bin'].value_counts().sort_index().to_string())


---
## Stage 2 — PrexSyn Seeds & Baseline Quality

**Pseudocode**: `seed_i = argmax Tanimoto(c, spec_i.fp)` over 256 candidates.
`baseline_quality_i = Tanimoto(seed_i, spec_i.fp)`.

The seed Tanimoto defines difficulty: specs with low `baseline_quality` are
harder for PrexSyn — modification may add more value in those bins.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(seeds_df['baseline_quality'], bins=20, edgecolor='white')
for x in (0.5, 0.7, 0.85):
    ax.axvline(x, color='gray', linestyle='--', linewidth=0.8)
ax.set_xlabel('Baseline quality (Tanimoto seed to spec)')
ax.set_ylabel('Specs')
ax.set_title('PrexSyn seed quality distribution')
plt.tight_layout()
plt.savefig(OUT_DIR / 'baseline_quality_dist.png', dpi=150)
plt.show()
print(seeds_df['baseline_quality'].describe().round(3).to_string())


---
## Stage 3 — Load Method Variants

**Pseudocode**: `variants_M_i = M.modify(seed_i, num_variants=N)`.

Each method notebook writes a scored CSV.  We load all registered methods;
missing files are skipped with a warning.

In [ ]:
per_method: dict[str, pd.DataFrame] = {}

for name, cfg in METHODS.items():
    path = cfg['scores']
    if not path.exists():
        print(f'[SKIP] {name}: {path.name} not found')
        continue
    df = pd.read_csv(path)
    df['method'] = name       # normalise label
    per_method[name] = df
    print(f'[OK]   {name}: {len(df):,} rows')

assert per_method, 'No method data found. Run method notebooks first.'


---
## Stage 4 — Evaluation Gates

**Pseudocode**:
```
# Gate 1: Synthesizability
if not AiZynthFinder.route_found(v): discard v

# Gate 2: Property conservation
substruct_v    = ECFP4_Tanimoto(v, spec_i.fp)               # tau_t
desirability_v = GeometricMean(per-descriptor tolerances)   # tau_d

if desirability_v >= tau_d AND substruct_v >= tau_t: mark v as HIT
```

`is_synth` is joined from each method's AiZynthFinder checkpoint.
Hit classification is re-applied uniformly at all `TAU_T_LIST` values.

In [ ]:
# Gate 1: attach is_synth from each checkpoint
for name, df in per_method.items():
    ckpt_path = METHODS[name]['synth_ckpt']
    if ckpt_path.exists():
        with open(ckpt_path) as f:
            ckpt: dict[str, bool] = json.load(f)
        df['is_synth'] = df['variant_smiles'].map(ckpt).fillna(False)
        print(f'{name}: {df["is_synth"].sum()}/{len(df)} synthesizable')
    else:
        # Baseline: only synth-passing variants were scored -> all True
        df['is_synth'] = True
        print(f'{name}: no checkpoint -> is_synth=True (baseline assumption)')

# Merge; restrict to synthesizable variants for all metrics
all_df   = pd.concat(per_method.values(), ignore_index=True)
synth_df = all_df[all_df['is_synth']].reset_index(drop=True)

print(f'\nAll variants  : {len(all_df):,}')
print(f'Synthesizable : {len(synth_df):,} ({100*len(synth_df)/max(len(all_df),1):.1f}%)')
print(synth_df['method'].value_counts().to_string())


---
## Stage 4b — Property Gate First, Then Synthesizability

**Optimized gate ordering**: run the cheap property-conservation filter
before the expensive AiZynthFinder MCTS, reducing the retrosynthesis
workload to only the candidates that already conserve the target profile.



Final hit sets are identical (the two gates are independent).
The cell quantifies the workload reduction and verifies equivalence.

In [ ]:
# Property gate at every tau_t -- no AiZynthFinder yet.
# Screen at the most permissive threshold so no hits are missed.
TAU_T_SCREEN = min(TAU_T_LIST)

prop_pass = all_df[
    (all_df["tanimoto"]     >= TAU_T_SCREEN) &
    (all_df["desirability"] >= TAU_D)
].copy().reset_index(drop=True)

print(f"Property gate (tau_t>={TAU_T_SCREEN}, tau_d>={TAU_D}) workload reduction:")
print(f"  {'Method':<12}  {'All':>7}  {'Prop-pass':>9}  {'Reduction':>10}")
for name in per_method:
    n_all  = int((all_df["method"] == name).sum())
    n_prop = int((prop_pass["method"] == name).sum())
    pct    = 100.0 * (1 - n_prop / max(n_all, 1))
    print(f"  {name:<12}  {n_all:>7,}  {n_prop:>9,}  {pct:>9.1f}%")

# Attach is_synth to the property-passing subset only.
# AiZynthFinder would run on prop_pass, not all_df.
for name in per_method:
    ckpt_path = METHODS[name]["synth_ckpt"]
    mask = prop_pass["method"] == name
    if ckpt_path.exists():
        with open(ckpt_path) as _f:
            ckpt = json.load(_f)
        prop_pass.loc[mask, "is_synth"] = (
            prop_pass.loc[mask, "variant_smiles"].map(ckpt).fillna(False)
        )
    else:
        prop_pass.loc[mask, "is_synth"] = True   # baseline: already synth-filtered

# Final hits: property gate AND synthesizable
optimized_df = prop_pass[prop_pass["is_synth"]].reset_index(drop=True)

# Verify equivalence with Stage 4 hit set
hits_4  = set(synth_df.loc[
    classify_hits(synth_df,     tau_t=TAU_T_SCREEN, tau_d=TAU_D), "variant_smiles"
])
hits_4b = set(optimized_df.loc[
    classify_hits(optimized_df, tau_t=TAU_T_SCREEN, tau_d=TAU_D), "variant_smiles"
])

print(f"\nHit sets identical : {hits_4 == hits_4b}")
print(f"Stage 4  synth calls: {len(all_df):,}  (all variants)")
print(f"Stage 4b synth calls: {len(prop_pass):,}  (property-passing only)")
saving = 100.0 * (1 - len(prop_pass) / max(len(all_df), 1))
print(f"Reduction           : {saving:.1f}% fewer retrosynthesis queries")


---
## Stage 5a — Per-Method Metrics (Table 1)

**Pseudocode**:
```
HitRate_M    = |hits_M| / |synthesizable_variants_M|
UniqueHits_M = |deduplicated hits_M|
Expansion_M  = UniqueHits_M / UniqueHits_baseline
Diversity_M  = 1 - MeanPairwiseTanimoto(hits_M)
```

`summarize()` computes all metrics at every `tau_t`, stratified by `quality_bin`.

In [ ]:
summary_df = summarize(synth_df, tau_t_list=TAU_T_LIST, tau_d=TAU_D)
summary_df.to_csv(SUMMARY_CSV, index=False)
print(f'Table 1 -> {SUMMARY_CSV}  ({len(summary_df)} rows)')
display(summary_df.sort_values(['tau_t', 'quality_bin', 'method']))


---
## Stage 5b — Stratification by Quality Bin

**Pseudocode**: `for each bin b: report metrics restricted to specs in b`

Hit rates by bin reveal when modification adds the most marginal value.

In [ ]:
BINS_ORDERED = ['<0.5', '0.5-0.7', '0.7-0.85', '0.85-1.0']
n_thresh = len(TAU_T_LIST)

fig, axes = plt.subplots(1, n_thresh, figsize=(5 * n_thresh, 4), sharey=True)
if n_thresh == 1:
    axes = [axes]

for ax, tau_t in zip(axes, TAU_T_LIST):
    sub = summary_df[summary_df['tau_t'] == tau_t].copy()
    sub['quality_bin'] = pd.Categorical(sub['quality_bin'], categories=BINS_ORDERED, ordered=True)
    pivot = sub.pivot(index='quality_bin', columns='method', values='hit_rate').reindex(BINS_ORDERED)
    pivot.plot.bar(ax=ax, width=0.7, edgecolor='white')
    ax.set_title(f'tau_t = {tau_t}')
    ax.set_xlabel('Baseline quality bin')
    if ax is axes[0]:
        ax.set_ylabel('Hit rate')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=8)

plt.suptitle('Hit rate by quality bin', y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'hit_rate_by_bin.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Score distributions (tanimoto + desirability) per method
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

for method, grp in synth_df.groupby('method'):
    if len(grp) > 1:
        grp['tanimoto'].plot.kde(ax=axes[0], label=method)
        grp['desirability'].plot.kde(ax=axes[1], label=method)

axes[0].axvline(TAU_T_LIST[0], color='gray', linestyle='--', linewidth=0.8, label='tau_t')
axes[0].set_xlabel('Tanimoto to spec')
axes[0].set_title('Substructural conservation')
axes[0].legend(fontsize=8)

axes[1].axvline(TAU_D, color='gray', linestyle='--', linewidth=0.8, label='tau_d')
axes[1].set_xlabel('Desirability')
axes[1].set_title('Physicochemical conservation')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / 'score_distributions.png', dpi=150)
plt.show()


In [ ]:
# Bioactive proximity: Tanimoto(variant, ChEMBL reference) -- diagnostic only.
# Not a success criterion; shows how close variants are to known bioactive space.
if 'bioactive_proximity' in synth_df.columns:
    fig, ax = plt.subplots(figsize=(5, 3))
    for method, grp in synth_df.groupby('method'):
        grp['bioactive_proximity'].dropna().plot.kde(ax=ax, label=method)
    ax.set_xlabel('Tanimoto to ChEMBL reference (diagnostic)')
    ax.set_title('Bioactive proximity (diagnostic only)')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'bioactive_proximity.png', dpi=150)
    plt.show()
else:
    print('bioactive_proximity not in data -- skipping diagnostic plot.')


---
## Stage 5c — Complementarity (Table 2)

**Pseudocode**:
```
for each pair (M_i, M_j):
    overlap_ij = |hits_i AND hits_j| / |hits_i OR hits_j|   # Jaccard

for each triple (M_i, M_j, M_k):
    combined = |hits_i OR hits_j OR hits_k|
```

Low Jaccard = complementary (non-overlapping coverage).

In [ ]:
TAU_T_PILOT = TAU_T_LIST[0]   # most permissive threshold for overlap analysis

hit_sets: dict[str, set[str]] = {}
for method, grp in synth_df.groupby('method'):
    mask = classify_hits(grp, tau_t=TAU_T_PILOT, tau_d=TAU_D)
    hit_sets[method] = set(grp.loc[mask, 'variant_smiles'])
    print(f'{method:12s}  unique hits: {len(hit_sets[method])}')

if len(hit_sets) >= 2:
    comp_df = complementarity(hit_sets)
    comp_df.to_csv(COMPLEMENT_CSV)
    print(f'\nPairwise Jaccard (tau_t={TAU_T_PILOT}, tau_d={TAU_D}):')
    display(comp_df.round(3))
else:
    print('Need >= 2 methods for complementarity analysis.')
    comp_df = pd.DataFrame()


In [ ]:
if not comp_df.empty:
    fig, ax = plt.subplots(figsize=(max(4, len(hit_sets)), max(3, len(hit_sets))))
    sns.heatmap(
        comp_df.astype(float), annot=True, fmt='.2f',
        cmap='YlOrRd', vmin=0, vmax=1, linewidths=0.5, ax=ax,
    )
    ax.set_title(f'Pairwise Jaccard overlap (tau_t={TAU_T_PILOT}, tau_d={TAU_D})')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'complementarity_heatmap.png', dpi=150)
    plt.show()

if len(hit_sets) >= 2:
    combos = best_combinations(hit_sets, top_k=10)
    combos_df = pd.DataFrame([
        {'methods': ' + '.join(c), 'unique_hits': n}
        for c, n in combos
    ])
    combos_df.to_csv(COMBOS_CSV, index=False)
    print(f'Top combinations (tau_t={TAU_T_PILOT}):')
    display(combos_df)


---
## Outputs

In [ ]:
print('=' * 55)
print('Pipeline Analysis -- outputs')
print('=' * 55)
for p in sorted(OUT_DIR.iterdir()):
    print(f'  {p.name:<40s} {p.stat().st_size/1e3:6.1f} kB')
